# Desafio — Fine-tuning de um ViT para classificação de imagens

Este notebook resolve o desafio:

> **Como usar um ViT (Visual Transformer) para realizar fine-tuning e criar um classificador para uma base de imagens com 10 classes?**

A estratégia adotada aqui usa **PyTorch + Hugging Face Transformers**, porque essa combinação costuma ser mais simples e estável para fine-tuning de modelos ViT em Colab do que a versão TensorFlow/Keras.

A base está no Google Drive:

```text
https://drive.google.com/drive/folders/1_V4gOqh-Cyqo4T6eZFLw-ztQhTDevZ-v?usp=sharing
```

## Estrutura esperada da base

O notebook assume que as imagens estão organizadas no formato abaixo:

```text
dataset/
  classe_1/
    img001.jpg
    img002.jpg
  classe_2/
    img001.jpg
    img002.jpg
  ...
  classe_10/
    img001.jpg
```

Cada subpasta representa uma classe. Esse é o formato usado pelo `torchvision.datasets.ImageFolder`.

## Conceito necessário

O **Vision Transformer (ViT)** adapta a arquitetura Transformer para imagens. Em vez de tratar a entrada como uma sequência de palavras, o ViT divide a imagem em pequenos **patches**. Cada patch é transformado em um vetor, de forma semelhante a um token em NLP.

Para uma imagem de tamanho 224 × 224 e patches de 16 × 16, o número de patches é:

\[
N = \frac{224}{16} \times \frac{224}{16} = 14 \times 14 = 196
\]

Esses 196 patches são processados pelo Transformer. No final, uma cabeça de classificação é ajustada para prever uma das classes da base.

Como a base possui aproximadamente 1000 imagens e 10 classes, a estratégia mais adequada é **fine-tuning** a partir de um modelo pré-treinado, e não treinamento do zero.

## Fórmulas principais

### Número de patches

\[
N = \frac{H}{P} \times \frac{W}{P}
\]

Onde:

- \(H\) = altura da imagem;
- \(W\) = largura da imagem;
- \(P\) = tamanho do patch.

### Função de perda para classificação multiclasse

Como o problema possui 10 classes, usamos **Cross-Entropy Loss**:

\[
L = -\sum_{i=1}^{C} y_i \log(\hat{y}_i)
\]

Onde:

- \(C\) é o número de classes;
- \(y_i\) é o rótulo real codificado em one-hot;
- \(\hat{y}_i\) é a probabilidade predita para a classe \(i\).

### Métricas de avaliação

Além da acurácia, serão avaliados:

- **precision** por classe;
- **recall** por classe;
- **F1-score** por classe;
- **F1 macro**, importante quando há desequilíbrio entre classes;
- matriz de confusão.

# 1. Instalação e imports

Execute esta célula no Colab. Depois, se o ambiente pedir, reinicie a sessão e execute novamente a partir dos imports.

In [ ]:
!pip install -q "transformers==4.38.2" accelerate datasets evaluate scikit-learn matplotlib pandas pillow gdown torchvision

In [ ]:
import os
import random
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch.utils.data import Dataset
from torchvision.datasets import ImageFolder
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer
)

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# 2. Acesso à base no Google Drive

Há duas formas de acessar a base:

1. **Download direto com `gdown`**, usando o link público da pasta.
2. **Montagem do Google Drive**, caso o download direto não funcione.

Primeiro, tente a opção com `gdown`. Se ela falhar, use a opção alternativa com `drive.mount`.

In [ ]:
# Opção A: baixar a pasta pública do Google Drive com gdown

DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1_V4gOqh-Cyqo4T6eZFLw-ztQhTDevZ-v?usp=sharing"
DATA_DIR = Path("/content/vit_dataset")

# Se quiser baixar novamente do zero, deixe True.
# Se a base já estiver baixada, deixe False.
FORCE_DOWNLOAD = False

if FORCE_DOWNLOAD and DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)

if not DATA_DIR.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print("Baixando base do Google Drive...")
    cmd = ["gdown", "--folder", DRIVE_FOLDER_URL, "-O", str(DATA_DIR), "--remaining-ok"]
    subprocess.run(cmd, check=False)
else:
    print("Pasta já existe:", DATA_DIR)

print("Arquivos/pastas encontrados em DATA_DIR:")
for item in DATA_DIR.iterdir():
    print("-", item)

## Opção alternativa: montar o Google Drive manualmente

Use esta opção se a célula anterior não conseguir baixar a pasta pública.

1. Abra o link da base no navegador.
2. Adicione um atalho da pasta ao seu Google Drive.
3. Monte o Drive no Colab.
4. Ajuste a variável `DATA_DIR` para o caminho da pasta.

In [ ]:
# Opção B: montar Google Drive manualmente
# Execute apenas se a opção com gdown não funcionar.

# from google.colab import drive
# drive.mount('/content/drive')

# Exemplo: ajuste conforme o local real da sua pasta no Drive
# DATA_DIR = Path('/content/drive/MyDrive/NOME_DA_PASTA_DA_BASE')
# print(DATA_DIR)
# print(list(DATA_DIR.iterdir())[:10])

# 3. Localizar automaticamente a pasta no formato ImageFolder

Às vezes, o download do Drive cria uma subpasta dentro de `/content/vit_dataset`. A função abaixo procura automaticamente a pasta que contém subpastas de classes com imagens.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

def count_images_in_folder(folder: Path):
    return sum(1 for f in folder.rglob("*") if f.suffix.lower() in IMAGE_EXTENSIONS)

def find_imagefolder_root(base_dir: Path, min_classes=2):
    candidates = []
    search_dirs = [base_dir] + [p for p in base_dir.rglob("*") if p.is_dir()]

    for p in search_dirs:
        subdirs = [d for d in p.iterdir() if d.is_dir()]
        if len(subdirs) < min_classes:
            continue

        class_counts = {d.name: count_images_in_folder(d) for d in subdirs}
        valid_counts = {k: v for k, v in class_counts.items() if v > 0}

        if len(valid_counts) >= min_classes:
            total = sum(valid_counts.values())
            candidates.append((p, total, len(valid_counts), valid_counts))

    if not candidates:
        raise FileNotFoundError(
            "Não encontrei uma pasta no formato ImageFolder. "
            "Verifique se a base está organizada como dataset/classe/imagem.jpg."
        )

    candidates = sorted(candidates, key=lambda x: x[1], reverse=True)
    return candidates[0]

DATA_ROOT, total_images, n_classes_found, class_counts = find_imagefolder_root(DATA_DIR)

print("Pasta raiz detectada:", DATA_ROOT)
print("Total de imagens:", total_images)
print("Número de classes detectadas:", n_classes_found)
print("Distribuição por classe:")
for cls, count in sorted(class_counts.items()):
    print(f"{cls}: {count}")

# 4. Carregar a base com ImageFolder e inspecionar classes

O `ImageFolder` interpreta cada subpasta como uma classe e atribui um índice numérico para cada classe.

In [ ]:
base_dataset = ImageFolder(root=str(DATA_ROOT))

class_names = base_dataset.classes
class_to_idx = base_dataset.class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}

print("Classes:")
for i, cls in enumerate(class_names):
    print(i, "->", cls)

print("\nTotal de imagens:", len(base_dataset))

In [ ]:
# Distribuição das classes
labels = [label for _, label in base_dataset.samples]
counts = pd.Series(labels).map(idx_to_class).value_counts().sort_index()

class_distribution = pd.DataFrame({
    "classe": counts.index,
    "quantidade": counts.values,
    "proporcao": counts.values / counts.values.sum()
})

class_distribution

In [ ]:
plt.figure(figsize=(10, 4))
plt.bar(class_distribution["classe"], class_distribution["quantidade"])
plt.xticks(rotation=45, ha="right")
plt.xlabel("Classe")
plt.ylabel("Quantidade de imagens")
plt.title("Distribuição de imagens por classe")
plt.tight_layout()
plt.show()

In [ ]:
# Visualizar algumas imagens aleatórias
n_samples = min(12, len(base_dataset))
indices = random.sample(range(len(base_dataset)), n_samples)

plt.figure(figsize=(12, 8))
for i, idx in enumerate(indices):
    img, label = base_dataset[idx]
    plt.subplot(3, 4, i + 1)
    plt.imshow(img)
    plt.title(idx_to_class[label])
    plt.axis("off")
plt.tight_layout()
plt.show()

# 5. Divisão treino / validação / teste

Será usada divisão estratificada:

- **70% treino**
- **15% validação**
- **15% teste**

A estratificação mantém a proporção das classes em cada subconjunto.

In [ ]:
samples = base_dataset.samples
paths = [p for p, y in samples]
y = [label for p, label in samples]

# Primeiro: separa treino de temporário, mantendo 30% para validação + teste
train_samples, temp_samples, y_train, y_temp = train_test_split(
    samples,
    y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE
)

# Segundo: divide o temporário ao meio: 15% validação, 15% teste
val_samples, test_samples, y_val, y_test = train_test_split(
    temp_samples,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

print("Treino:", len(train_samples))
print("Validação:", len(val_samples))
print("Teste:", len(test_samples))

In [ ]:
def distribution_from_samples(samples, name):
    labels = [label for _, label in samples]
    counts = pd.Series(labels).map(idx_to_class).value_counts().sort_index()
    return pd.DataFrame({
        "conjunto": name,
        "classe": counts.index,
        "quantidade": counts.values,
        "proporcao": counts.values / counts.values.sum()
    })

pd.concat([
    distribution_from_samples(train_samples, "treino"),
    distribution_from_samples(val_samples, "validacao"),
    distribution_from_samples(test_samples, "teste")
], ignore_index=True)

# 6. Preparar o ViT e o dataset PyTorch

Modelo usado:

```text
google/vit-base-patch16-224-in21k
```

Esse checkpoint é adequado para fine-tuning porque já foi pré-treinado em uma grande base de imagens. A cabeça de classificação será substituída para o número de classes da nossa base.

In [ ]:
MODEL_CHECKPOINT = "google/vit-base-patch16-224-in21k"

image_processor = AutoImageProcessor.from_pretrained(MODEL_CHECKPOINT)

id2label = {i: cls for i, cls in enumerate(class_names)}
label2id = {cls: i for i, cls in enumerate(class_names)}

model = AutoModelForImageClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(class_names),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

print("Número de classes:", len(class_names))
print("Modelo carregado:", MODEL_CHECKPOINT)

In [ ]:
class ViTImageDataset(Dataset):
    def __init__(self, samples, image_processor, augment=False):
        self.samples = samples
        self.image_processor = image_processor
        self.augment = augment

        self.train_augment = transforms.Compose([
            transforms.RandomResizedCrop(224, scale=(0.80, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, label = self.samples[idx]
        image = Image.open(image_path).convert("RGB")

        if self.augment:
            image = self.train_augment(image)

        encoding = self.image_processor(images=image, return_tensors="pt")
        pixel_values = encoding["pixel_values"].squeeze(0)

        return {
            "pixel_values": pixel_values,
            "labels": torch.tensor(label, dtype=torch.long)
        }

train_dataset = ViTImageDataset(train_samples, image_processor, augment=True)
val_dataset = ViTImageDataset(val_samples, image_processor, augment=False)
test_dataset = ViTImageDataset(test_samples, image_processor, augment=False)

print("Dataset de treino:", len(train_dataset))
print("Dataset de validação:", len(val_dataset))
print("Dataset de teste:", len(test_dataset))

In [ ]:
def collate_fn(batch):
    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    labels = torch.tensor([item["labels"].item() for item in batch], dtype=torch.long)
    return {
        "pixel_values": pixel_values,
        "labels": labels
    }

# Teste rápido de um batch
batch = collate_fn([train_dataset[i] for i in range(min(4, len(train_dataset)))])
print(batch["pixel_values"].shape)
print(batch["labels"])

# 7. Métricas de avaliação

Será usada a acurácia e também F1 macro/ponderado. O **F1 macro** é importante porque calcula a média do F1 de cada classe sem ponderar pela quantidade de exemplos, sendo útil quando há desequilíbrio entre as classes.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )

    return {
        "accuracy": acc,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
    }

# 8. Treinamento

Para uma base pequena, o treinamento deve ser conservador:

- poucas épocas;
- learning rate baixo;
- data augmentation leve;
- seleção do melhor modelo pelo F1 macro na validação.

Se ocorrer erro de memória, reduza `per_device_train_batch_size` para 4.

In [ ]:
OUTPUT_DIR = "/content/vit_finetuned_image_classifier"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=10,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=8,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    remove_unused_columns=False,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=RANDOM_STATE
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=image_processor,
    data_collator=collate_fn,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

# 9. Avaliação no conjunto de validação e teste

Após o treinamento, o conjunto de validação serve para acompanhar o ajuste do modelo, e o conjunto de teste serve para a avaliação final.

In [ ]:
val_metrics = trainer.evaluate(val_dataset)
print("Métricas de validação:")
val_metrics

In [ ]:
test_metrics = trainer.evaluate(test_dataset)
print("Métricas de teste:")
test_metrics

In [ ]:
pred_output = trainer.predict(test_dataset)
logits = pred_output.predictions
y_true = pred_output.label_ids
y_pred = np.argmax(logits, axis=1)

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    digits=4,
    zero_division=0
))

# 10. Matriz de confusão

A matriz de confusão mostra quais classes o modelo mais confunde entre si.

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(10, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, xticks_rotation=45, values_format="d")
plt.title("Matriz de Confusão — ViT")
plt.tight_layout()
plt.show()

# 11. Exemplos classificados incorretamente

Esta etapa ajuda a responder se os erros do modelo fazem sentido visualmente. É importante verificar se há imagens ambíguas, baixa qualidade, objetos parecidos ou rótulos possivelmente inconsistentes.

In [ ]:
# Probabilidades por softmax
probs = torch.nn.functional.softmax(torch.tensor(logits), dim=1).numpy()
confidences = probs.max(axis=1)

error_rows = []
for i, ((path, true_label), pred_label, conf) in enumerate(zip(test_samples, y_pred, confidences)):
    error_rows.append({
        "arquivo": path,
        "classe_real": idx_to_class[true_label],
        "classe_predita": idx_to_class[pred_label],
        "correto": true_label == pred_label,
        "confianca": conf
    })

df_predicoes = pd.DataFrame(error_rows)
df_erros = df_predicoes[df_predicoes["correto"] == False].copy()

print("Total de exemplos de teste:", len(df_predicoes))
print("Total de erros:", len(df_erros))
print("Taxa de erro:", len(df_erros) / len(df_predicoes))

df_erros.sort_values("confianca", ascending=False).head(20)

In [ ]:
def mostrar_erros(df_erros, n=12):
    if len(df_erros) == 0:
        print("Não houve erros no conjunto de teste.")
        return

    amostra = df_erros.sample(min(n, len(df_erros)), random_state=RANDOM_STATE)

    plt.figure(figsize=(14, 10))
    for i, (_, row) in enumerate(amostra.iterrows()):
        img = Image.open(row["arquivo"]).convert("RGB")
        plt.subplot(3, 4, i + 1)
        plt.imshow(img)
        plt.title(
            f"Real: {row['classe_real']}\nPred: {row['classe_predita']}\nConf: {row['confianca']:.2f}",
            fontsize=9
        )
        plt.axis("off")
    plt.tight_layout()
    plt.show()

mostrar_erros(df_erros, n=12)

# 12. Salvar o modelo treinado

Esta etapa salva o modelo e o processador de imagem para uso posterior.

In [ ]:
FINAL_MODEL_DIR = "/content/vit_modelo_final"

trainer.save_model(FINAL_MODEL_DIR)
image_processor.save_pretrained(FINAL_MODEL_DIR)

print("Modelo salvo em:", FINAL_MODEL_DIR)

In [ ]:
# Opcional: compactar o modelo para download
!zip -r /content/vit_modelo_final.zip /content/vit_modelo_final > /dev/null
print("Arquivo ZIP criado em: /content/vit_modelo_final.zip")

# 13. Discussão dos resultados — preencher após executar

> Substitua os campos entre colchetes pelos resultados reais do seu notebook.

O experimento realizou o **fine-tuning de um Vision Transformer (ViT)** para uma tarefa de **classificação multiclasse de imagens**. A base foi organizada em **10 classes**, sendo cada subpasta interpretada como uma classe pelo carregador `ImageFolder`.

O modelo utilizado foi o `google/vit-base-patch16-224-in21k`, pré-treinado em uma grande base de imagens. Para este problema, a cabeça final de classificação foi substituída por uma nova camada com **10 saídas**, correspondentes às classes da base. As imagens foram redimensionadas e normalizadas pelo `AutoImageProcessor`, e foi aplicado **data augmentation leve** no conjunto de treino para reduzir o risco de sobreajuste.

A divisão dos dados foi feita em três conjuntos:

- treino: **[N_TREINO]** imagens;
- validação: **[N_VALIDACAO]** imagens;
- teste: **[N_TESTE]** imagens.

No conjunto de teste, o modelo obteve:

- acurácia: **[ACURACIA_TESTE]**;
- F1 macro: **[F1_MACRO_TESTE]**;
- F1 ponderado: **[F1_WEIGHTED_TESTE]**.

A acurácia indica a proporção geral de imagens classificadas corretamente. No entanto, como pode haver diferenças na quantidade de imagens por classe, o **F1 macro** é uma métrica especialmente importante, pois avalia o desempenho médio entre as classes sem favorecer classes com mais exemplos.

A análise da matriz de confusão mostrou que as maiores confusões ocorreram entre as classes **[CLASSE_A]** e **[CLASSE_B]**, e também entre **[CLASSE_C]** e **[CLASSE_D]**. Isso sugere que essas classes possuem características visuais semelhantes ou que o conjunto de treinamento não possui exemplos suficientes para separar bem esses padrões.

A inspeção dos exemplos classificados incorretamente mostrou que alguns erros fazem sentido visualmente. Em alguns casos, as imagens parecem ambíguas, apresentam baixa resolução, objetos parcialmente visíveis, fundo com ruído ou semelhança visual entre classes. Esses fatores podem dificultar a decisão do modelo.

Durante o treinamento, deve-se observar se a perda de treino continua caindo enquanto a perda de validação aumenta. Caso isso ocorra, há indício de **sobreajuste**. Nesse caso, recomenda-se reduzir o número de épocas, aumentar regularização, aplicar mais data augmentation ou congelar parte do backbone nas primeiras épocas.

Conclui-se que o ViT é uma estratégia adequada para o desafio porque aproveita conhecimento prévio aprendido em grandes bases de imagens e adapta esse conhecimento para a base específica por meio de fine-tuning. O desempenho final deve ser interpretado com base não apenas na acurácia, mas também no F1-score por classe, na matriz de confusão e na análise qualitativa dos erros.

# 14. Resposta final esperada — preencher após executar

O Vision Transformer foi ajustado para a tarefa de **classificação de imagens em 10 classes**. O modelo recebeu imagens pré-processadas, dividiu cada imagem em patches e extraiu representações visuais contextuais por meio da arquitetura Transformer. Em seguida, uma cabeça de classificação foi treinada para prever a classe de cada imagem.

O desempenho final no conjunto de teste foi de **[ACURACIA_TESTE]** de acurácia, com **[F1_MACRO_TESTE]** de F1 macro e **[F1_WEIGHTED_TESTE]** de F1 ponderado.

A matriz de confusão mostrou que o modelo classificou melhor as classes **[LISTAR_CLASSES_COM_MELHOR_DESEMPENHO]** e teve maior dificuldade nas classes **[LISTAR_CLASSES_COM_PIOR_DESEMPENHO]**. Os principais erros ocorreram entre **[PARES_DE_CLASSES_MAIS_CONFUNDIDAS]**, possivelmente devido à semelhança visual entre essas categorias, baixa quantidade de exemplos ou variações nas imagens.

Portanto, o fine-tuning com ViT mostrou-se uma abordagem viável para o problema. Como melhoria futura, recomenda-se testar mais data augmentation, ajustar o número de épocas, avaliar congelamento parcial do backbone e comparar o ViT com uma CNN pré-treinada, como ResNet ou EfficientNet.

# Referências técnicas usadas no notebook

- Hugging Face Transformers — documentação do ViT.
- Hugging Face Transformers — tarefa de classificação de imagens.
- Hugging Face Trainer — API de treinamento em PyTorch.
- PyTorch/Torchvision — `ImageFolder` para bases organizadas por pastas de classe.